# Set up library

In [27]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint,  EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
import h5py
from sklearn.utils.class_weight import compute_class_weight
# from autolrtuner import AutoLRTuner
import os
# from audiomentations import Compose, AddGaussianNoise, Gain





# Mount Drive

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Main

### encoding labels

In [29]:
HDF5_file_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5/Qeej_Hmong_Features.h5"
y = None
if os.path.exists(HDF5_file_path):
      print("Retrieving data from drive with path:", HDF5_file_path)
      with h5py.File(HDF5_file_path, "r") as f:
        y = f["labels"][:]
        y = [name.decode('utf-8') for name in y]

le = LabelEncoder()
le.fit_transform(y)
class_names = le.classes_
print("check class_names", class_names)

Retrieving data from drive with path: /content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5/Qeej_Hmong_Features.h5
check class_names ['P1' 'P1_P2' 'P1_P2_P3' 'P1_P2_P3_T1' 'P1_P2_P3_T1_T2'
 'P1_P2_P3_T1_T2_T3' 'P1_P2_P3_T1_T3' 'P1_P2_P3_T2' 'P1_P2_P3_T2_T3'
 'P1_P2_P3_T3' 'P1_P2_T1' 'P1_P2_T1_T2' 'P1_P2_T1_T2_T3' 'P1_P2_T1_T3'
 'P1_P2_T2' 'P1_P2_T2_T3' 'P1_P2_T3' 'P1_P3' 'P1_P3_T1' 'P1_P3_T1_T2'
 'P1_P3_T1_T2_T3' 'P1_P3_T1_T3' 'P1_P3_T2' 'P1_P3_T2_T3' 'P1_P3_T3'
 'P1_T1' 'P1_T1_T2' 'P1_T1_T2_T3' 'P1_T1_T3' 'P1_T2' 'P1_T2_T3' 'P1_T3'
 'drone' 'drone_P2' 'drone_P2_P3' 'drone_P2_P3_T1' 'drone_P2_P3_T1_T2'
 'drone_P2_P3_T1_T2_T3' 'drone_P2_P3_T1_T3' 'drone_P2_P3_T2'
 'drone_P2_P3_T2_T3' 'drone_P2_P3_T3' 'drone_P2_T1' 'drone_P2_T1_T2'
 'drone_P2_T1_T2_T3' 'drone_P2_T1_T3' 'drone_P2_T2' 'drone_P2_T2_T3'
 'drone_P2_T3' 'drone_P3' 'drone_P3_T1' 'drone_P3_T1_T2'
 'drone_P3_T1_T2_T3' 'drone_P3_T1_T3' 'drone_P3_T2' 'drone_P3_T2_T3'
 'drone_P3_T3' 'drone_T1' 'drone_T1_T2' 'drone_T1_T2_T3' 'drone_T1_T3'
 

In [30]:
Model_Path = "/content/drive/MyDrive/Qeej_Hmong_Model_Parameters/CheckPoints/model_latest (1).keras"
model = load_model(Model_Path)

In [31]:
def extract_dft(window):
  eps = 1e-10
  spectrum = np.abs(np.fft.rfft(window))
  spectrum = spectrum / np.max(spectrum + eps)
  return spectrum

def suppress_low_amplitudes(spec, threshold_db=50):
  I_threshold = 10 ** (-threshold_db / 20)
  spec[spec < I_threshold] *= 0.2
  return spec
def get_spectrum_of_window(window):

  spectrum = extract_dft(window)

  #preprocessing
  standard_spectrum = suppress_low_amplitudes(spectrum)

  return standard_spectrum

In [32]:
def get_spectrograms_of_audio(audio, samples_per_window):
        X = []
        # preparing data
        for i in np.arange(0, len(audio), samples_per_window):
            start_sample = i
            end_sample = i + samples_per_window
            window = audio[start_sample:end_sample]
            spectrum = get_spectrum_of_window(window)

            X.append(spectrum)


        return np.array(X)


In [33]:
import json
def assess_audio(
        audio_path,
        sample_rate=48000,
        window_size=0.01
    ):

        #load audio
        audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)

        print("check len audio: ", len(audio))

        samples_per_window = int(sample_rate * window_size)
        print("samples per window: ", samples_per_window)

        #get features X
        X = get_spectrograms_of_audio(audio, samples_per_window)

        # predictions results
        y_pred = model.predict(X[..., np.newaxis])
        y_pred = np.argmax(y_pred, axis=1)
        y_to_str = le.inverse_transform(y_pred)

        print("result of audio: ", audio_path)
        print("check len of labels", len(y_to_str))
        print(y_to_str)


        #check the most appear
        unique, counts = np.unique(y_to_str, return_counts=True)

        data = dict(zip(unique.tolist(), counts.tolist()))

        print(json.dumps(data, indent=4))






## assess all audio

In [34]:
 # read data from drive
audio_data_set_path = "/content/drive/MyDrive/Real Data/Data"

if os.path.exists(audio_data_set_path):
  for root, dirs, files in os.walk(audio_data_set_path):
    for file in files:
      if file.endswith(".wav"):
        audio_file_path = os.path.join(root, file)
        assess_audio(audio_file_path)

else:
  print("Cant find folder at /content/drive/MyDrive/Qeej_Hmong_Dataset/Data/audio")


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
    "P1_P2_T1": 1,
    "P1_P2_T1_T3": 1,
    "P1_P3_T1_T2": 5,
    "P1_P3_T1_T2_T3": 3,
    "P1_P3_T1_T3": 1,
    "P1_P3_T3": 11,
    "drone": 9,
    "drone_P2_P3_T1_T2": 2,
    "drone_P2_P3_T1_T2_T3": 1,
    "drone_P2_P3_T3": 1,
    "drone_P2_T1_T2": 1,
    "drone_P2_T2": 2,
    "drone_P2_T2_T3": 1,
    "drone_P3": 35,
    "drone_P3_T1": 2,
    "drone_P3_T1_T2": 11,
    "drone_P3_T1_T2_T3": 3,
    "drone_P3_T2_T3": 2,
    "drone_P3_T3": 7,
    "drone_T1_T2_T3": 1,
    "drone_T2": 1,
    "drone_T3": 1,
    "silence": 3
}
check len audio:  45120
samples per window:  480
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
result of audio:  /content/drive/MyDrive/Real Data/Data/Khèn 3/Đơn âm/Thổi ra/Thổi ra 1 trái_2.wav
check len of labels 94
['silence' 'drone_T1' 'drone_P2_P3_T1_T2' 'drone_T1' 'drone_T1' 'drone_T1'
 'drone_T1' 'drone_T1' 'drone_T1' 'drone_T1' 'drone_T1' 'drone_T1'
 'drone_T1' 'drone_T1' 'drone_T1' 'drone_T1' 'drone_T1' '